# Dependencia y concentración de las importaciones de Bogotá
### Bogotá DataJam 2026 · Equipo **S3D.ETL** · Fundación Universitaria Compensar

**Pregunta de análisis.** ¿Qué tan concentradas están las importaciones con destino a Bogotá por país, región del mundo y nivel tecnológico, y cómo cambió su valor entre el primer bimestre de 2025 y el de 2026 cuando se mide en dólares y en pesos colombianos?

**Hipótesis.** Las importaciones de Bogotá dependen de pocos socios (China y Estados Unidos) y de bienes manufacturados de tecnología media y alta. El crecimiento en dólares entre 2025 y 2026 no se traduce en un mayor costo en pesos, porque la apreciación del peso (TRM más baja) lo compensa.

**Fuentes de datos públicas integradas**

| # | Conjunto de datos | Entidad | Uso en el análisis | Llave de integración |
|---|---|---|---|---|
| 1 | Importaciones Bogotá | Secretaría Distrital de Desarrollo Económico (Portal Datos Abiertos Bogotá) | Hechos: valor CIF/FOB, peso, país, partida, CIIU, nivel tecnológico | — |
| 2 | Tasa de Cambio Representativa del Mercado (TRM) | Superintendencia Financiera de Colombia (datos.gov.co) | Convertir el valor importado a pesos | Año + mes |
| 3 | ISO 3166 con regiones de la ONU (M49) | División de Estadística de la ONU / ISO (repositorio público en GitHub) | Región y subregión de cada país de origen | País → código ISO alfa-3 |
| 3b | Nombres de países en español (ISO 3166) | Repositorio público *world_countries* | Puente entre nombres en español y códigos ISO | Nombre normalizado |

**Estructura del notebook**
1. Extracción de las fuentes
2. Limpieza y transformación (réplica en Python del ETL hecho en Power Query y OpenRefine)
3. Integración de fuentes
4. Indicadores y resultados
5. Exportación (datos procesados para Power BI, tablas, figuras y dashboard HTML)

In [1]:
# Librerías y rutas (todas relativas a la raíz del repositorio)
from pathlib import Path
import unicodedata
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, EXT, PROC = RAIZ / "data/raw", RAIZ / "data/external", RAIZ / "data/processed"
FIG, TAB, OUT = RAIZ / "outputs/figuras", RAIZ / "outputs/tablas", RAIZ / "outputs"
for p in (PROC, FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 60)
print("Raíz del repositorio:", RAIZ)

Raíz del repositorio: /tmp/claude-0/-home-claude/afd0fa51-ad3e-5e3d-99f1-0a80d61b8956/scratchpad/repo/datajam-importaciones-bogota


## 1. Extracción

**Fuente 1 – Importaciones Bogotá.** Archivo original descargado del portal (https://datosabiertos.bogota.gov.co/dataset/importaciones-bogota) el 20 de agosto de 2026. Se guarda en `data/raw` como CSV comprimido, sin ninguna modificación, para que el repositorio sea liviano y reproducible.

In [2]:
orig = pd.read_csv(RAW / "importaciones_bogota_original.csv.gz", encoding="utf-8")
print(f"Registros originales: {len(orig):,} | Columnas: {orig.shape[1]}")
orig.head(3)

Registros originales: 132,289 | Columnas: 17


,Periodo,Mes,Codigo pais origen,Pais de origen,Codigo partida,Nombre partida,Codigo ciiu rev4,Descripcion ciiu rev4,Codigo capitulo,Nombre capitulo,Nivel tecnologico,Uso economico,Detalle uso economico,Dolares CIF,Dolares FOB,Kilogramos netos,Cantidad
0,2025,1,23,Alemania,1210,"Conos de lúpul o frescos o secos, incluso triturados, mo...",128,Cultivo de especias y de plantas aromáticas y medicinales,12,"Semillas y frutos oleaginosos, forrajes",Bienes primarios,Materias primas,Productos alimenticios,4533.66,3740.45,220.0,220.0
1,2025,1,23,Alemania,1901,"Extracto de malta; preparaciones alimenticias de harina,...",1051,Elaboración de productos de molinería,19,"Preparaciones a base de cereal, harina, leche; pastelería",Manufacturas basadas en recursos naturales,Bienes de consumo,Alimentos,2777.71,1676.31,300.0,300.0
2,2025,1,23,Alemania,1901,"Extracto de malta; preparaciones alimenticias de harina,...",1089,Elaboración de otros productos alimenticios n.c.p.,19,"Preparaciones a base de cereal, harina, leche; pastelería",Manufacturas basadas en recursos naturales,Bienes de consumo,Alimentos,1590.58,1515.15,900.0,900.0


**Fuente 2 – TRM (Superintendencia Financiera, datos.gov.co).** Se consulta la API Socrata del conjunto `32sa-8pi3` y se calcula la TRM promedio de cada mes. Si no hay conexión, se usa la copia guardada en `data/external/trm_mensual_2025_2026.csv`, que se obtuvo con la misma consulta.

In [3]:
URL_TRM = ("https://www.datos.gov.co/resource/32sa-8pi3.json?"
           "$select=date_trunc_ym(vigenciadesde)%20as%20mes,avg(valor)%20as%20trm"
           "&$where=vigenciadesde%20between%20'2025-01-01T00:00:00'%20and%20'2026-03-01T00:00:00'"
           "&$group=mes&$order=mes")
try:
    import json, urllib.request
    with urllib.request.urlopen(URL_TRM, timeout=30) as r:
        t = pd.DataFrame(json.load(r))
    trm = pd.DataFrame({"anio": pd.to_datetime(t["mes"]).dt.year,
                        "mes": pd.to_datetime(t["mes"]).dt.month,
                        "trm_promedio": t["trm"].astype(float)})
    trm.to_csv(EXT / "trm_mensual_2025_2026.csv", index=False)
    print("TRM descargada desde datos.gov.co")
except Exception as e:
    trm = pd.read_csv(EXT / "trm_mensual_2025_2026.csv")
    print("Sin conexión a datos.gov.co, se usa la copia local:", type(e).__name__)
trm.round(2)

Sin conexión a datos.gov.co, se usa la copia local: URLError


,anio,mes,trm_promedio
0,2025,1,4292.27
1,2025,2,4133.60
2,2025,3,4133.48
3,2025,4,4273.88
4,2025,5,4201.50
5,2025,6,4113.55
6,2025,7,4049.69
7,2025,8,4049.93
8,2025,9,3914.83
9,2025,10,3881.44


**Fuente 3 – Regiones del mundo (ISO 3166 / ONU M49)** y **3b – nombres de países en español.** Se descargan de sus repositorios públicos, con copia local de respaldo.

In [4]:
FUENTES_GH = {
    "iso3166_regiones_onu_m49.csv": "https://raw.githubusercontent.com/lukes/ISO-3166-Countries-with-Regional-Codes/master/all/all.csv",
    "paises_nombres_es_iso.csv": "https://raw.githubusercontent.com/stefangabos/world_countries/master/data/countries/es/world.csv",
}
tablas = {}
for archivo, url in FUENTES_GH.items():
    try:
        tablas[archivo] = pd.read_csv(url, keep_default_na=False)
        tablas[archivo].to_csv(EXT / archivo, index=False)
        print("Descargado:", archivo)
    except Exception as e:
        tablas[archivo] = pd.read_csv(EXT / archivo, keep_default_na=False)
        print("Copia local:", archivo, type(e).__name__)
iso = tablas["iso3166_regiones_onu_m49.csv"]
nombres_es = tablas["paises_nombres_es_iso.csv"]
equiv = pd.read_csv(EXT / "equivalencias_paises_manual.csv")
print(f"ISO/M49: {len(iso)} países | nombres ES: {len(nombres_es)} | equivalencias manuales: {len(equiv)}")

Descargado: iso3166_regiones_onu_m49.csv
Descargado: paises_nombres_es_iso.csv
ISO/M49: 249 países | nombres ES: 249 | equivalencias manuales: 26


## 2. Limpieza y transformación

Se replican en Python las decisiones documentadas en el primer corte (Power Query + OpenRefine), para que cualquier persona pueda reproducirlas:

| Paso | Problema detectado | Decisión |
|---|---|---|
| 2.1 | Códigos de país, partida, CIIU y capítulo tipados como número (se pierden ceros a la izquierda) | Convertir a texto y completar ceros |
| 2.2 | 9 filas duplicadas exactas | Eliminar |
| 2.3 | 65 filas con FOB = 0 pero CIF > 0 y 4 filas con 0 kg | Convertir a nulo (dato faltante, no cero real) |
| 2.4 | 7 códigos CIIU con más de una descripción | Unificar: quitar puntuación final y usar la descripción real en vez de «Sin información» |
| 2.5 | «Sin información» / «Sin Información» usados como categoría | Nivel tecnológico → nulo; uso económico → «Sin clasificar» |
| 2.6 | La columna país mezcla países y zonas francas | Nueva columna *Tipo de origen* |
| 2.7 | Fecha inexistente (solo año y mes) | Crear fecha del primer día del mes |

In [5]:
df = orig.copy()
diagnostico = {"registros_originales": len(df)}

# 2.1 Códigos como texto (identificadores, no cantidades)
df["Codigo pais origen"] = df["Codigo pais origen"].astype(str)
df["Codigo partida"] = df["Codigo partida"].astype(str).str.zfill(4)
df["Codigo ciiu rev4"] = df["Codigo ciiu rev4"].astype(str).str.zfill(4)
df["Codigo capitulo"] = df["Codigo capitulo"].astype(str).str.zfill(2)

# 2.2 Duplicados exactos
diagnostico["duplicados_exactos"] = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

# 2.3 Ceros que en realidad son datos faltantes
diagnostico["fob_cero"] = int((df["Dolares FOB"] == 0).sum())
diagnostico["kg_cero"] = int((df["Kilogramos netos"] == 0).sum())
df.loc[df["Dolares FOB"] == 0, "Dolares FOB"] = np.nan
df.loc[df["Kilogramos netos"] == 0, "Kilogramos netos"] = np.nan
diagnostico

{'registros_originales': 132289,
 'duplicados_exactos': 9,
 'fob_cero': 65,
 'kg_cero': 4}

In [6]:
# 2.4 Descripciones CIIU inconsistentes
SIN_INFO = {"sin información", "sin informacion"}
antes = df.groupby("Codigo ciiu rev4")["Descripcion ciiu rev4"].nunique()
print("Códigos CIIU con más de una descripción (original):", int((antes > 1).sum()))
desc = df["Descripcion ciiu rev4"].str.strip().str.rstrip(".")

validas = df.assign(d=desc)[~desc.str.lower().isin(SIN_INFO)]
canon = validas.groupby("Codigo ciiu rev4")["d"].agg(lambda s: s.value_counts().index[0])
df["Descripcion ciiu rev4"] = df["Codigo ciiu rev4"].map(canon).fillna("Sin clasificar")
print("Después de unificar:", int(df.groupby("Codigo ciiu rev4")["Descripcion ciiu rev4"].nunique().max()), "descripción por código")

# 2.5 «Sin información» no es una categoría válida
es_sin = lambda s: s.str.strip().str.lower().isin(SIN_INFO)
diagnostico["nivel_tec_sin_info"] = int(es_sin(df["Nivel tecnologico"]).sum())
diagnostico["uso_eco_sin_info"] = int(es_sin(df["Uso economico"]).sum())
df.loc[es_sin(df["Nivel tecnologico"]), "Nivel tecnologico"] = np.nan
df.loc[es_sin(df["Uso economico"]), "Uso economico"] = "Sin clasificar"

# 2.6 Tipo de origen (país vs zona franca colombiana)
df["Tipo origen"] = np.where(df["Pais de origen"].str.contains("zona franca", case=False), "Zona franca", "País")

# 2.7 Fecha
df["Fecha"] = pd.to_datetime(dict(year=df["Periodo"], month=df["Mes"], day=1))

# Validación lógica: CIF debe ser >= FOB (se tolera 1 centavo por redondeo de coma flotante)
df["Validacion"] = np.where(df["Dolares FOB"] - df["Dolares CIF"] > 0.01, "Revisar", "OK")
diagnostico.update({"fob_mayor_cif": int((df["Validacion"] == "Revisar").sum()),
                    "registros_limpios": len(df),
                    "zonas_francas": int((df["Tipo origen"] == "Zona franca").sum())})
pd.Series(diagnostico, name="valor").to_frame()

Códigos CIIU con más de una descripción (original): 7


Después de unificar: 1 descripción por código


,valor
registros_originales,132289
duplicados_exactos,9
fob_cero,65
kg_cero,4
nivel_tec_sin_info,6137
uso_eco_sin_info,298
fob_mayor_cif,0
registros_limpios,132280
zonas_francas,621


**Hallazgo de calidad.** La columna `Mes` solo contiene los valores 1 y 2. El conjunto publicado cubre **enero–febrero de 2025 y enero–febrero de 2026**, no una serie mensual continua como sugiere su descripción. Por eso el análisis compara bimestres equivalentes.

In [7]:
df.groupby(["Periodo", "Mes"]).size().rename("registros").to_frame()

registros
Periodo Mes           
2025    1        33073
        2        32638
2026    1        33555
        2        33014

## 3. Integración de fuentes

### 3.1 Importaciones + regiones del mundo (llave: país → ISO alfa-3)
Los nombres del dataset están en español y no siguen un estándar. Se normalizan (minúsculas, sin tildes) y se cruzan con la tabla de nombres ISO en español. Los casos que no coinciden (variantes como «Tunicia» o «Belarus», o nombres truncados) se resuelven con una tabla de equivalencias manual documentada en `data/external/equivalencias_paises_manual.csv`.

In [8]:
def normalizar(s):
    s = unicodedata.normalize("NFKD", str(s).lower())
    return "".join(c for c in s if not unicodedata.combining(c)).strip()

nombres_es["k"] = nombres_es["name"].map(normalizar)
nombres_es["alpha3"] = nombres_es["alpha3"].str.upper()
puente = nombres_es.drop_duplicates("k").set_index("k")["alpha3"]

paises = df[["Pais de origen", "Tipo origen"]].drop_duplicates().copy()
paises["alpha3"] = paises["Pais de origen"].map(normalizar).map(puente)
manual = equiv.set_index("pais_origen_dataset")["alpha3"]
paises["alpha3"] = paises["alpha3"].fillna(paises["Pais de origen"].map(manual))
paises["metodo_cruce"] = np.select(
    [paises["Tipo origen"] == "Zona franca", paises["Pais de origen"].isin(manual.index), paises["alpha3"].notna()],
    ["Zona franca (Colombia)", "Equivalencia manual", "Coincidencia automática"], "Sin identificar")

# Regiones de la ONU traducidas al español
REG = {"Africa": "África", "Americas": "América", "Asia": "Asia", "Europe": "Europa", "Oceania": "Oceanía"}
SUB = {"Northern America": "Norteamérica", "South America": "Suramérica", "Central America": "Centroamérica y México",
       "Caribbean": "Caribe", "Eastern Asia": "Asia Oriental", "South-eastern Asia": "Sudeste Asiático",
       "Southern Asia": "Asia Meridional", "Western Asia": "Asia Occidental", "Central Asia": "Asia Central",
       "Western Europe": "Europa Occidental", "Southern Europe": "Europa Meridional", "Northern Europe": "Europa del Norte",
       "Eastern Europe": "Europa Oriental", "Northern Africa": "África del Norte", "Sub-Saharan Africa": "África Subsahariana",
       "Australia and New Zealand": "Australia y Nueva Zelanda", "Melanesia": "Melanesia", "Micronesia": "Micronesia",
       "Polynesia": "Polinesia", "Eastern Africa": "África Oriental", "Middle Africa": "África Central",
       "Southern Africa": "África Austral", "Western Africa": "África Occidental"}
iso2 = iso.rename(columns={"alpha-3": "alpha3"}).copy()
iso2["Region"] = iso2["region"].map(REG)
iso2["Subregion"] = np.where(iso2["intermediate-region"] != "", iso2["intermediate-region"], iso2["sub-region"])
iso2["Subregion"] = iso2["Subregion"].map(SUB).fillna(iso2["Subregion"])
paises = paises.merge(iso2[["alpha3", "Region", "Subregion"]], on="alpha3", how="left")
paises.loc[paises["Tipo origen"] == "Zona franca", ["Region", "Subregion"]] = ["Colombia (zonas francas)", "Zonas francas de Colombia"]
paises[["Region", "Subregion"]] = paises[["Region", "Subregion"]].fillna("Sin identificar")

df = df.merge(paises[["Pais de origen", "alpha3", "Region", "Subregion", "metodo_cruce"]], on="Pais de origen", how="left")
cobertura = df.groupby("metodo_cruce")["Dolares CIF"].agg(registros="size", cif_usd="sum")
cobertura["% del CIF"] = (cobertura["cif_usd"] / cobertura["cif_usd"].sum() * 100).round(2)
cobertura

,registros,cif_usd,% del CIF
metodo_cruce,,,
Coincidencia automática,126416,2.157068e+10,96.52
Equivalencia manual,5171,2.542425e+08,1.14
Sin identificar,72,1.335922e+08,0.60
Zona franca (Colombia),621,3.905262e+08,1.75


### 3.2 Importaciones + TRM (llave: año + mes)
Cada registro se valora en pesos colombianos multiplicando su valor CIF por la TRM promedio de su mes. Así se puede separar el efecto del volumen comprado del efecto de la tasa de cambio.

In [9]:
df = df.merge(trm.rename(columns={"anio": "Periodo", "mes": "Mes", "trm_promedio": "TRM"}), on=["Periodo", "Mes"], how="left")
assert df["TRM"].notna().all(), "Hay meses sin TRM"
df["Pesos CIF"] = df["Dolares CIF"] * df["TRM"]
df[["Periodo", "Mes", "Pais de origen", "Region", "Dolares CIF", "TRM", "Pesos CIF"]].head()

,Periodo,Mes,Pais de origen,Region,Dolares CIF,TRM,Pesos CIF
0,2025,1,Alemania,Europa,4533.66,4292.266842,1.945968e+07
1,2025,1,Alemania,Europa,2777.71,4292.266842,1.192267e+07
2,2025,1,Alemania,Europa,1590.58,4292.266842,6.827194e+06
3,2025,1,Alemania,Europa,63574.89,4292.266842,2.728804e+08
4,2025,1,Alemania,Europa,15327.32,4292.266842,6.578895e+07


## 4. Indicadores y resultados

In [10]:
ok = df[df["Validacion"] == "OK"]  # excluye registros con FOB > CIF (en esta versión: ninguno)
def var(a, b): return (b - a) / a * 100

res = ok.groupby("Periodo").agg(cif_usd=("Dolares CIF", "sum"), fob_usd=("Dolares FOB", "sum"),
                                cif_cop=("Pesos CIF", "sum"), kg=("Kilogramos netos", "sum"), registros=("Dolares CIF", "size"))
res["trm_promedio_bimestre"] = ok.groupby("Periodo").apply(lambda g: g["Pesos CIF"].sum() / g["Dolares CIF"].sum())
res["usd_por_kg"] = res["cif_usd"] / res["kg"]
variacion = (res.loc[2026] / res.loc[2025] - 1) * 100
resumen = pd.concat([res.T, variacion.rename("Var. %")], axis=1)
resumen.columns = ["Ene-Feb 2025", "Ene-Feb 2026", "Var. %"]
resumen.to_csv(TAB / "resumen_interanual.csv")
resumen.style.format("{:,.2f}")

,Ene-Feb 2025,Ene-Feb 2026,Var. %
cif_usd,"10,704,695,503.90","11,644,345,906.79",8.78
fob_usd,"10,076,661,130.85","11,028,058,403.08",9.44
cif_cop,"45,102,349,193,041.36","42,921,418,260,198.40",-4.84
kg,"7,118,670,647.01","7,420,472,480.83",4.24
registros,"65,711.00","66,569.00",1.31
trm_promedio_bimestre,"4,213.32","3,686.03",-12.51
usd_por_kg,1.50,1.57,4.35


**Concentración por país – Índice Herfindahl-Hirschman (IHH).** IHH = Σ (participación %)². Por debajo de 1.500 el mercado se considera poco concentrado, entre 1.500 y 2.500 moderadamente concentrado y por encima de 2.500 altamente concentrado.

In [11]:
paises_anio = ok[ok["Tipo origen"] == "País"].pivot_table(index="Pais de origen", columns="Periodo", values="Dolares CIF", aggfunc="sum", fill_value=0)
part = paises_anio / paises_anio.sum() * 100
ihh = (part ** 2).sum().round(0)
top = part.sort_values(2026, ascending=False)
conc = pd.DataFrame({"IHH": ihh, "Top 1 %": top.iloc[0], "Top 2 %": top.iloc[:2].sum(), "Top 5 %": top.iloc[:5].sum(),
                     "Países proveedores": (paises_anio > 0).sum()}).round(1)
conc.to_csv(TAB / "concentracion_por_pais.csv")
print("Mayores proveedores 2026:", ", ".join(top.index[:5]))
conc

Mayores proveedores 2026: China, Estados Unidos, México, Brasil, Alemania


,IHH,Top 1 %,Top 2 %,Top 5 %,Países proveedores
Periodo,,,,,
2025,1505.0,28.0,53.0,65.3,154
2026,1556.0,31.3,53.1,65.8,159


In [12]:
top10 = ok.groupby("Pais de origen").agg(CIF_USD=("Dolares CIF", "sum"), Region=("Region", "first")).sort_values("CIF_USD", ascending=False)
top10["Part. %"] = top10["CIF_USD"] / top10["CIF_USD"].sum() * 100
v = paises_anio.assign(var_pct=lambda x: var(x[2025], x[2026]))["var_pct"]
top10["Var. 26/25 %"] = v
top10 = top10.head(10)
top10.to_csv(TAB / "top10_paises.csv")
top10.style.format({"CIF_USD": "{:,.0f}", "Part. %": "{:.1f}", "Var. 26/25 %": "{:+.1f}"})

,CIF_USD,Region,Part. %,Var. 26/25 %
Pais de origen,,,,
China,"6,521,457,715",Asia,29.2,+22.0
Estados Unidos,"5,131,228,415",América,23.0,-4.7
México,"1,059,326,329",América,4.7,+17.9
Brasil,"974,452,883",América,4.4,+4.6
Alemania,"713,681,935",Europa,3.2,+14.9
India,"569,211,594",Asia,2.5,+6.7
Japón,"548,272,209",Asia,2.5,+34.2
Corea del Sur,"481,831,497",Asia,2.2,+32.0
Italia,"393,868,969",Europa,1.8,+3.7


In [13]:
reg = ok.pivot_table(index="Region", columns="Periodo", values="Dolares CIF", aggfunc="sum", fill_value=0)
reg["Part. 2026 %"] = reg[2026] / reg[2026].sum() * 100
reg["Var. %"] = var(reg[2025], reg[2026])
reg = reg.sort_values(2026, ascending=False)
reg.to_csv(TAB / "cif_por_region.csv")
sub = ok.groupby("Subregion")["Dolares CIF"].sum().sort_values(ascending=False)
(sub / sub.sum() * 100).head(8).round(1).to_csv(TAB / "top_subregiones.csv")
reg.style.format("{:,.1f}")

Periodo,2025,2026,Part. 2026 %,Var. %
Region,,,,
Asia,"4,164,412,185.9","5,051,274,345.2",43.4,21.3
América,"4,699,234,993.7","4,671,690,589.0",40.1,-0.6
Europa,"1,470,238,432.1","1,558,293,552.2",13.4,6.0
Colombia (zonas francas),"203,780,960.5","186,745,274.4",1.6,-8.4
Sin identificar,"118,138,568.7","133,504,692.7",1.1,13.0
África,"39,862,017.1","34,010,975.9",0.3,-14.7
Oceanía,"9,028,345.8","8,826,477.5",0.1,-2.2


In [14]:
niv = ok.assign(Nivel=ok["Nivel tecnologico"].fillna("Sin clasificar")).pivot_table(
    index="Region", columns="Nivel", values="Dolares CIF", aggfunc="sum", fill_value=0)
niv_pct = niv.div(niv.sum(axis=1), axis=0) * 100
niv_pct = niv_pct.loc[reg.index]
niv_pct.to_csv(TAB / "nivel_tecnologico_por_region.csv")
alta_total = niv["Manufacturas de alta tecnología"].sum() / niv.values.sum() * 100
print(f"Participación de alta tecnología en el total: {alta_total:.1f} %")
niv_pct.round(1)

Participación de alta tecnología en el total: 18.7 %


Nivel,Bienes primarios,Manufacturas basadas en recursos naturales,Manufacturas de alta tecnología,Manufacturas de baja tecnología,Manufacturas de tecnología media,Otras transacciones,Sin clasificar
Region,,,,,,,
Asia,1.4,10.4,23.8,22.0,39.6,0.1,2.7
América,25.5,31.0,10.0,5.9,26.3,0.3,1.1
Europa,4.9,15.0,31.8,7.9,37.7,0.4,2.3
Colombia (zonas francas),1.8,74.7,8.3,4.6,10.4,0.0,0.2
Sin identificar,0.3,10.9,19.2,7.8,60.3,0.0,1.4
África,13.1,25.2,1.1,11.7,46.8,0.1,2.0
Oceanía,13.4,7.5,18.5,17.5,38.8,1.3,2.9


In [15]:
uso = ok.pivot_table(index="Uso economico", columns="Periodo", values="Dolares CIF", aggfunc="sum")
uso["Part. 2026 %"] = uso[2026] / uso[2026].sum() * 100
uso["Var. %"] = var(uso[2025], uso[2026])
uso.to_csv(TAB / "cif_por_uso_economico.csv")
cap = ok.groupby(["Codigo capitulo", "Nombre capitulo"])["Dolares CIF"].sum().sort_values(ascending=False).head(10)
cap.to_csv(TAB / "top10_capitulos.csv")
uso.style.format("{:,.1f}")

Periodo,2025,2026,Part. 2026 %,Var. %
Uso economico,,,,
Bienes de capital,"2,742,675,949.3","3,183,821,124.9",27.3,16.1
Bienes de consumo,"2,636,270,646.5","3,365,645,645.2",28.9,27.7
Materias primas,"5,324,105,712.3","5,092,665,877.9",43.7,-4.3
Sin clasificar,"1,643,195.9","2,213,258.8",0.0,34.7


### Figuras

In [16]:
AZUL, AZUL2, NAR, GRIS = "#1B3A6B", "#2F7AC4", "#E0712C", "#9AA5B4"
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

# F1: dólares vs pesos
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for a, col, tit, esc, uni in [(ax[0], "cif_usd", "Valor CIF en dólares", 1e9, "miles de millones USD"),
                              (ax[1], "cif_cop", "Valor CIF en pesos (con TRM)", 1e12, "billones COP")]:
    vals = res[col] / esc
    b = a.bar(["Ene-Feb 2025", "Ene-Feb 2026"], vals, color=[GRIS, AZUL2])
    a.bar_label(b, fmt="%.2f"); a.set_title(f"{tit}\n{variacion[col]:+.1f} %", fontsize=11); a.set_ylabel(uni)
fig.suptitle(f"TRM promedio: {res.loc[2025,'trm_promedio_bimestre']:,.0f} → {res.loc[2026,'trm_promedio_bimestre']:,.0f} COP/USD ({variacion['trm_promedio_bimestre']:+.1f} %)", y=1.02)
fig.tight_layout(); fig.savefig(FIG / "01_usd_vs_cop.png", dpi=150, bbox_inches="tight"); plt.show()

# F2: top 10 países
fig, ax = plt.subplots(figsize=(8, 4.5))
t = top10.iloc[::-1]
ax.barh(t.index, t["CIF_USD"] / 1e6, color=[NAR if i >= 8 else AZUL2 for i in range(10)])
for i, (vv, p) in enumerate(zip(t["CIF_USD"] / 1e6, t["Part. %"])): ax.text(vv, i, f" {vv:,.0f} M ({p:.1f} %)", va="center", fontsize=9)
ax.set_xlabel("CIF (millones de USD), Ene-Feb 2025 + 2026"); ax.set_title("Top 10 países de origen: China y EE. UU. concentran más de la mitad")
fig.tight_layout(); fig.savefig(FIG / "02_top10_paises.png", dpi=150); plt.show()

# F3: nivel tecnológico por región
fig, ax = plt.subplots(figsize=(9, 4.2))
orden_niv = ["Manufacturas de alta tecnología", "Manufacturas de tecnología media", "Manufacturas de baja tecnología",
             "Manufacturas basadas en recursos naturales", "Bienes primarios", "Otras transacciones", "Sin clasificar"]
cols = ["#1B3A6B", "#2F7AC4", "#6AA6DD", "#A9CBEC", "#E0712C", "#C4C9D1", "#E5E7EB"]
bottom = np.zeros(len(niv_pct))
for n, c in zip(orden_niv, cols):
    if n in niv_pct:
        ax.barh(niv_pct.index, niv_pct[n], left=bottom, color=c, label=n.replace("Manufacturas ", "")); bottom += niv_pct[n].values
ax.invert_yaxis(); ax.set_xlabel("% del CIF de la región"); ax.legend(ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.18), loc="upper center", frameon=False)
ax.set_title("Composición tecnológica de las importaciones por región de origen")
fig.tight_layout(); fig.savefig(FIG / "03_nivel_tec_por_region.png", dpi=150, bbox_inches="tight"); plt.show()

## 5. Exportación

In [17]:
# Datos integrados para Power BI (CSV comprimido, UTF-8, punto decimal)
columnas = ["Periodo", "Mes", "Fecha", "Codigo pais origen", "Pais de origen", "Tipo origen", "alpha3", "Region", "Subregion",
        "Codigo partida", "Nombre partida", "Codigo ciiu rev4", "Descripcion ciiu rev4", "Codigo capitulo", "Nombre capitulo",
        "Nivel tecnologico", "Uso economico", "Detalle uso economico", "Dolares CIF", "Dolares FOB", "Kilogramos netos",
        "Cantidad", "TRM", "Pesos CIF", "Validacion"]
df[columnas].to_csv(PROC / "importaciones_integrado.csv.gz", index=False, encoding="utf-8", compression="gzip", date_format="%Y-%m-%d")
paises.to_csv(PROC / "dim_pais_region.csv", index=False, encoding="utf-8")
print("Exportado:", PROC / "importaciones_integrado.csv.gz", f"({len(df):,} filas, {len(columnas)} columnas)")

Exportado: /tmp/claude-0/-home-claude/afd0fa51-ad3e-5e3d-99f1-0a80d61b8956/scratchpad/repo/datajam-importaciones-bogota/data/processed/importaciones_integrado.csv.gz (132,280 filas, 25 columnas)


In [18]:
# Dashboard HTML interactivo (se abre en cualquier navegador)
vista = ok.assign(Nivel=ok["Nivel tecnologico"].fillna("Sin clasificar"), Anio=ok["Periodo"].astype(str))
f = make_subplots(rows=3, cols=2, specs=[[{"type": "xy"}, {"type": "xy"}], [{"type": "domain"}, {"type": "xy"}], [{"type": "treemap", "colspan": 2}, None]],
                  subplot_titles=("CIF por bimestre: USD vs COP (índice 2025 = 100)", "Top 10 países por CIF (millones USD)",
                                  "Participación por región de origen", "Nivel tecnológico por región (% del CIF)",
                                  "Capítulos arancelarios (clic para ampliar)"), vertical_spacing=0.1, row_heights=[0.28, 0.32, 0.40])
idx = res[["cif_usd", "cif_cop"]] / res[["cif_usd", "cif_cop"]].iloc[0] * 100
for c, n, col in [("cif_usd", "USD", AZUL2), ("cif_cop", "COP", NAR)]:
    f.add_bar(x=["Ene-Feb 2025", "Ene-Feb 2026"], y=idx[c], name=n, marker_color=col, text=idx[c].round(1), row=1, col=1)
f.add_bar(y=top10.index[::-1], x=(top10["CIF_USD"] / 1e6)[::-1], orientation="h", marker_color=AZUL, showlegend=False, row=1, col=2)
r2 = reg[2025] + reg[2026]
f.add_pie(labels=r2.index, values=r2.values, hole=0.5, showlegend=False, textinfo="label+percent", row=2, col=1)
base = np.zeros(len(niv_pct))
for n, c in zip(orden_niv, cols):
    if n in niv_pct:
        f.add_bar(y=niv_pct.index, x=niv_pct[n], base=base.copy(), orientation="h", offsetgroup="nivel",
                  name=n.replace("Manufacturas ", ""), marker_color=c, row=2, col=2)
        base += niv_pct[n].values
tm = ok.groupby(["Nombre capitulo", "Nombre partida"])["Dolares CIF"].sum().reset_index()
tm["Partida"] = tm["Nombre partida"].str.slice(0, 60)
tmf = px.treemap(tm, path=["Nombre capitulo", "Partida"], values="Dolares CIF")
f.add_trace(tmf.data[0], row=3, col=1)
f.update_layout(barmode="group", height=1500, template="plotly_white", font=dict(family="Segoe UI, Arial"),
                title=dict(text=(f"<b>Importaciones con destino a Bogotá · Ene–Feb 2025 vs 2026</b><br><sup>CIF {res['cif_usd'].sum()/1e9:,.1f} mil M USD · "
                                 f"Var. USD {variacion['cif_usd']:+.1f} % · Var. COP {variacion['cif_cop']:+.1f} % · IHH países {ihh[2026]:,.0f} · "
                                 "Fuentes: SDDE (Datos Abiertos Bogotá), Superfinanciera (TRM), ONU M49</sup>")),
                legend=dict(orientation="h", y=0.415, yanchor="top", x=0.5, xanchor="center", font=dict(size=10)))
f.update_yaxes(autorange="reversed", row=2, col=2)
f.write_html(OUT / "dashboard_importaciones_bogota.html", include_plotlyjs=True)
print("Dashboard:", OUT / "dashboard_importaciones_bogota.html")

Dashboard: /tmp/claude-0/-home-claude/afd0fa51-ad3e-5e3d-99f1-0a80d61b8956/scratchpad/repo/datajam-importaciones-bogota/outputs/dashboard_importaciones_bogota.html


## Conclusiones
Las cifras salen de las tablas de `outputs/tablas` y corresponden a Ene–Feb 2025 frente a Ene–Feb 2026.

1. **Alta y creciente concentración en Asia.** China (29,2 % del CIF acumulado) y Estados Unidos (23,0 %) aportan más de la mitad del valor importado. El IHH por país pasó de 1.505 a 1.556 y entró en el rango de concentración moderada (> 1.500). Asia creció 21,3 % y en 2026 superó a América como primera región de origen (43,4 % frente a 40,1 %). China creció 22,0 % y Estados Unidos cayó 4,7 %.
2. **Dependencia tecnológica.** El 33,5 % del valor son manufacturas de tecnología media y el 18,7 % de alta tecnología. Europa y Asia concentran las importaciones de alta tecnología.
3. **El tipo de cambio cambia la lectura.** El valor CIF creció 8,8 % en dólares, pero cayó 4,8 % en pesos, porque la TRM promedio bajó 12,5 % (de 4.213 a 3.686 COP/USD). Medir solo en dólares sobreestima el aumento del gasto de las empresas bogotanas.
4. **Cambio en la composición.** Los bienes de consumo crecieron 27,7 % y los de capital 16,1 %, mientras que las materias primas cayeron 4,3 %. Esto puede indicar un mayor peso del consumo importado frente a los insumos para la producción local.
5. **Calidad de los datos.** El dataset publicado cubre solo enero y febrero de cada año, mezcla zonas francas con países, usa «Sin información» como categoría y no estandariza los nombres de países. El 3,5 % del valor no se pudo cruzar automáticamente con códigos ISO.

**Recomendación.** La SDDE podría incorporar al seguimiento del comercio exterior distrital un tablero trimestral con el IHH por país y por región, el valor en pesos y la composición por uso económico, como alerta temprana de dependencia. También conviene publicar el dataset con código ISO del país y la serie mensual completa.